Name: **NITIN JAIN** |
Roll no: **M24AID022**



---
**Assignment 2** |
**Machine Learning for Big Data**

**Github Repository (https://github.com/nitinjain2015/M24AID022_CSL7110_Assignment2**)

## Q1: k-Grams, MinHash Implementation, Jaccard Similarities using given zip file (minhash.zip) having 4 documents (D1,D2,D3,D4)



In [1]:
# Upload the given zip file minhash.zip
from google.colab import files
uploaded = files.upload()

Saving minhash.zip to minhash.zip


In [2]:
# Extract the files from the uploaded zip file
import zipfile, os
with zipfile.ZipFile('minhash.zip', 'r') as zip_ref:
    zip_ref.extractall('minhash')
print(os.listdir('minhash'))

['minhash']


In [3]:
# Read the given 4 documents extracted from the zip file
def read_doc(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().strip()

D1 = read_doc('minhash/minhash/D1.txt')
D2 = read_doc('minhash/minhash/D2.txt')
D3 = read_doc('minhash/minhash/D3.txt')
D4 = read_doc('minhash/minhash/D4.txt')

docs = {'D1': D1, 'D2': D2, 'D3': D3, 'D4': D4}
print('Documents loaded')

Documents loaded


In [4]:
# k-gram functions
def char_kgrams(text, k):
    return {text[i:i+k] for i in range(len(text)-k+1)}

def word_kgrams(text, k):
    words = text.split()
    return {tuple(words[i:i+k]) for i in range(len(words)-k+1)}

In [5]:
# Construct k-grams
char2 = {name: char_kgrams(text, 2) for name, text in docs.items()}
char3 = {name: char_kgrams(text, 3) for name, text in docs.items()}
word2 = {name: word_kgrams(text, 2) for name, text in docs.items()}
print('k-grams constructed')

# Print the count of k-grams constructued for difefrent documents

print("\nCharacter 2-grams count:")
for doc in char2:
    print(doc, len(char2[doc]))

print("\nCharacter 3-grams count:")
for doc in char3:
    print(doc, len(char3[doc]))

print("\nWord 2-grams count:")
for doc in word2:
    print(doc, len(word2[doc]))


k-grams constructed

Character 2-grams count:
D1 263
D2 262
D3 269
D4 255

Character 3-grams count:
D1 765
D2 762
D3 828
D4 698

Word 2-grams count:
D1 279
D2 278
D3 337
D4 232


In [6]:
# Q1A Using character 3-grams build a min-hash signature for documents D1 and D2
#     using given values of t (20, 60, 150, 300, 600) for hash functions and
#     report approxiate Jaccard similarity between docs D1 and D2 (5 numders to
#     be reported)

# MinHash for D1-D2 using character 3-grams
import hashlib, random
random.seed(42) # taking initial seed as 42 to ensure output is considetent across runs

def gram_to_int(gram):
    return int(hashlib.md5(str(gram).encode()).hexdigest(), 16)

def generate_hash_functions(t, m=20000):
    return [(random.randint(1, m-1), random.randint(0, m-1)) for _ in range(t)]

def minhash_signature(grams, hash_funcs, m=20000):
    gram_ids = [gram_to_int(g) for g in grams]
    sig = []
    for (a,b) in hash_funcs:
        sig.append(min(((a*x + b) % m) for x in gram_ids))
    return sig

def signature_similarity(sig1, sig2):
    return sum(1 for i in range(len(sig1)) if sig1[i]==sig2[i]) / len(sig1)

# Given values of t
t_values = [20, 60, 150, 300, 600]
print('\nMinHash Approximation (D1-D2):')
for t in t_values:
    hash_funcs = generate_hash_functions(t)
    sig1 = minhash_signature(char3['D1'], hash_funcs)
    sig2 = minhash_signature(char3['D2'], hash_funcs)
    approx = signature_similarity(sig1, sig2)
    print(f"t={t} -> Approx Jaccard={approx:.4f}")



MinHash Approximation (D1-D2):
t=20 -> Approx Jaccard=1.0000
t=60 -> Approx Jaccard=0.9833
t=150 -> Approx Jaccard=0.9933
t=300 -> Approx Jaccard=0.9900
t=600 -> Approx Jaccard=0.9800


In [7]:
# Q1B  Calculate Exact Jaccard similarity between all pair of documents for
#      each type of k-gram ( 3 x 6 = 18 values to be reported)

from itertools import combinations

def jaccard(A, B):
    return len(A & B) / len(A | B)

def compute_all_jaccard(gram_dict, label):
    print(f"\n--- {label} ---")
    for d1, d2 in combinations(docs.keys(), 2):
        sim = jaccard(gram_dict[d1], gram_dict[d2])
        print(f"{d1}-{d2}: {sim:.4f}")

compute_all_jaccard(char2, 'Character 2-grams : Jaccard Similarity')
compute_all_jaccard(char3, 'Character 3-grams : Jaccard Similarity')
compute_all_jaccard(word2, 'Word 2-grams : Jaccard Similarity')


--- Character 2-grams : Jaccard Similarity ---
D1-D2: 0.9811
D1-D3: 0.8157
D1-D4: 0.6444
D2-D3: 0.8000
D2-D4: 0.6413
D3-D4: 0.6530

--- Character 3-grams : Jaccard Similarity ---
D1-D2: 0.9780
D1-D3: 0.5804
D1-D4: 0.3051
D2-D3: 0.5680
D2-D4: 0.3059
D3-D4: 0.3121

--- Word 2-grams : Jaccard Similarity ---
D1-D2: 0.9408
D1-D3: 0.1823
D1-D4: 0.0302
D2-D3: 0.1737
D2-D4: 0.0303
D3-D4: 0.0161


## Q2 MinHashing
## A. Using 3-grams build in Q1 to build min-hash signature for documents D1 and D2 using t =20,60,150,300,600 hash functions and calculate the estimated Jaccard similarity ( 5 numbers to be reported)

##B. What seems to be a good value for t, Justify answer in terms of both accuracy and time

In [8]:
## Q2A Import libraries, set the initial parameters and convert k garms to integers
import hashlib, random
import pandas as pd

random.seed(42) # Fixing the random seed so that output is consistent across runs
m = 20000

def gram_to_int(gram):

    return int(hashlib.md5(str(gram).encode()).hexdigest(),16)

In [9]:
## Generate Hash Functions
def generate_hash_functions(t):

    hash_funcs = []

    for _ in range(t):
        a = random.randint(1,m-1)
        b = random.randint(0,m-1)
        hash_funcs.append((a,b))
    return hash_funcs

In [10]:
## Define the signature function

def minhash_signature(grams, hash_funcs):
    gram_ids = [gram_to_int(g) for g in grams]
    signature = []

    for a,b in hash_funcs:
        min_hash = min((a*x + b) % m for x in gram_ids)
        signature.append(min_hash)
    return signature

In [11]:
## Define the signature similarity  function
def signature_similarity(sig1,sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i]==sig2[i])
    return matches/len(sig1)

In [12]:
# Execute for different vaues of t as given in question
# amd comparing approx Jaccard , error and runtime

import time
import pandas as pd

t_values = [20,60,150,300,600] # Given in question

exact_jaccard = 0.9780   # from Q1B

results = []

random.seed(42)

for t in t_values:

    start = time.time()

    hash_funcs = generate_hash_functions(t)

    sig1 = minhash_signature(char3["D1"], hash_funcs)
    sig2 = minhash_signature(char3["D2"], hash_funcs)

    approx = signature_similarity(sig1, sig2)

    end = time.time()

    runtime = end - start

    error = abs(exact_jaccard - approx)

    results.append([t, approx, exact_jaccard, error, runtime])

df_q2 = pd.DataFrame(results,
                     columns=["t (hash functions)",
                              "Approx Jaccard",
                              "Exact Jaccard",
                              "Absolute Error",
                              "Runtime (seconds)"])

df_q2.round(5)

,t (hash functions),Approx Jaccard,Exact Jaccard,Absolute Error,Runtime (seconds)
0,20,1.00000,0.978,0.02200,0.01391
1,60,0.98333,0.978,0.00533,0.03742
2,150,0.99333,0.978,0.01533,0.08383
3,300,0.99000,0.978,0.01200,0.15325
4,600,0.98000,0.978,0.00200,0.32590


**Q2B Answer : Good value of t and Justification** <br>
 MinHash estimate become closer to the exact Jaccard similarity as the no. of hash functions increase while the runtime increases approximately linearly with the number of hash functions. A value of t between 150 and 300 provides a good balance between accuracy and runtime.

Q3 **LSH :** Consider computing LSH using t -= 160 hash functions <br>
A.  Estimate the best values of hash functions(b) within each of the (r) bands for  the S-curve <br>
B. Using the choice of ( r ) and ( b ) and ( f(⋅) ), what is the probability of each pair of the four documents (using 3-grams) being estimated to have a similarity greater than ( τ )? Report 6 numbers.


Q3A Solution

The approximate threshold of the LSH S-curve is s ≈ (1/𝑏) ^ (1/𝑟)

Given t = 160 = r * b

With r=8 and b= 20

t = 8 x 20 = 160 and  threshold becomes
(1/20)^ 1/8≈0.69 which is close to the desired similarity threshold
τ=0.7

This combination of r = 8 and b = 20  produces an S-curve where document pairs with similarity near 0.7 have a high probability of becoming candidate pairs.

In [13]:
# Q 3A Code
#Import the Libraries
import pandas as pd
from itertools import combinations

# Define the Jacccard Function
def jaccard(A,B):
    return len(A & B) / len(A | B)

# Compute the Jaccard Similarity
pairs = []
for d1, d2 in combinations(docs.keys(),2):
    s = jaccard(char3[d1], char3[d2])
    pairs.append([d1+"-"+d2, s])
df_jaccard = pd.DataFrame(pairs,
                          columns=["Document Pair","Jaccard Similarity"])
df_jaccard.round(5)


,Document Pair,Jaccard Similarity
0,D1-D2,0.97798
1,D1-D3,0.58036
2,D1-D4,0.30508
3,D2-D3,0.56805
4,D2-D4,0.30590
5,D3-D4,0.31212


In [14]:
# Q 3B Code

# Define LSH Probability function
def lsh_probability(s,r,b):
    return 1 - (1 - s**r)**b

# Choose the LSH Parameters
t = 160  #(Given)
τ = 0.7  #(Given)
r = 8    # Explained in Q3A
b = 20   # Explained in Q3A

# Compute the probabilities

df_jaccard["LSH Probability"] = df_jaccard["Jaccard Similarity"].apply(
    lambda s: lsh_probability(s,r,b)
)
df_jaccard.round(5)

,Document Pair,Jaccard Similarity,LSH Probability
0,D1-D2,0.97798,1.00000
1,D1-D3,0.58036,0.22822
2,D1-D4,0.30508,0.00150
3,D2-D3,0.56805,0.19588
4,D2-D4,0.30590,0.00153
5,D3-D4,0.31212,0.00180


Q3B Results Conclusion  

*  D1–D2 pair has similarity greater than τ = 0.7, so it has probability ≈ 1 of being detected
*  D1–D3 and D2–D3 pairs have moderate similarity and moderate probability pf detection
*  All pairs involving D4 i.e. D1–D4, D2-D4 and D3–D4 have very low probabilities (~0.001), hence  they are unlikely to be candidate pairs.

Q4 Min-Hashing on MovieLens dataset
Implement Min-Hashing on the older MovieLens 100k dataset (5MB), which consists of a set of 943 users who have rated 1682 movies. You can download the data from:http://www.grouplens.org/node/73

Compute the exact Jaccard similarity for all pairs of users, and output the pairs of users that have a similarity of at least 0.5.

Compute the min-hash signatures for the users and compute the approximate Jaccard similarity using 50, 100 and 200 hash functions

For each value, output the pairs that have an estimated similarity of at least 0.5 and reportthe number of false positives and false negatives.
For the false positives and negatives, report the averages for 5 different runs.

In [15]:
#Q4 Solution
# Import the libraries
import pandas as pd
import random

from itertools import combinations

#Dowbload the desired older MovieLens 100k dataset (5MB), which consists of a set
#of 943 users who have rated 1682 movies.

!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -oq ml-100k.zip
print('Dataset dowmoaded successfully ')

--2026-03-06 10:04:32--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  3.59MB/s    in 1.3s    

2026-03-06 10:04:34 (3.59 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Dataset dowmoaded successfully 


In [16]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user","movie","rating","timestamp"]
)

print("Display first 5 rows")
display(ratings.head())

print("\nDataset Statistics")
print("Total ratings:", len(ratings))
print("Total users:", ratings["user"].nunique())
print("Total movies:", ratings["movie"].nunique())

Display first 5 rows


,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596



Dataset Statistics
Total ratings: 100000
Total users: 943
Total movies: 1682


In [17]:
# Build user-,movie sets
user_movies = ratings.groupby("user")["movie"].apply(set)

users = list(user_movies.index)

print("Total users:", len(users))

# for optrimization convert movie sets to list
user_movies_list = {
    u:list(movies)
    for u,movies in user_movies.items()
}

# Define Jaccard similarity
def jaccard(A,B):
    return len(A & B) / len(A | B)


# Generate hash functions
m = 20000
def generate_hash_functions(t):
    funcs = []

    for _ in range(t):
        a = random.randint(1,m-1)
        b = random.randint(0,m-1)
        funcs.append((a,b))
    return funcs

Total users: 943


In [18]:
# Define minhash signature function
def minhash_signature(movie_list, hash_funcs):
    sig = []

    for a,b in hash_funcs:
        min_val = float("inf")

        for movie in movie_list:
            h = (a*movie + b) % m
            if h < min_val:
                min_val = h
        sig.append(min_val)
    return sig

In [19]:
# Define signature similarity
def signature_similarity(sig1,sig2):

    matches = sum(
        1 for i in range(len(sig1))
        if sig1[i] == sig2[i]
    )
    return matches/len(sig1)

In [20]:
# Precompute all user pairs
user_pairs = list(combinations(users,2))

print("Total user pairs:", len(user_pairs))

Total user pairs: 444153


In [21]:
# Precompute exact similarities
exact_sim = {}

for u1,u2 in user_pairs:
    exact_sim[(u1,u2)] = jaccard(
        user_movies[u1],
        user_movies[u2]
    )

In [22]:
# Count true pairs >= 0.5
threshold = 0.5

true_pairs = sum(
    1 for s in exact_sim.values()
    if s >= threshold
)

print("True pairs with similarity ≥ 0.5:", true_pairs)

True pairs with similarity ≥ 0.5: 10


In [23]:
# Run the min hash experiments 5 times as per question
runs = 5
t_values = [50,100,200]

results = []

for t in t_values:

    fp_total = 0
    fn_total = 0

    for run in range(runs):

        random.seed(42 + run)

        hash_funcs = generate_hash_functions(t)

        signatures = {}

        for u in users:

            signatures[u] = minhash_signature(
                user_movies_list[u],
                hash_funcs
            )

        fp = 0
        fn = 0

        for u1,u2 in user_pairs:

            exact = exact_sim[(u1,u2)]

            approx = signature_similarity(
                signatures[u1],
                signatures[u2]
            )

            if approx >= threshold and exact < threshold:
                fp += 1

            if approx < threshold and exact >= threshold:
                fn += 1

        fp_total += fp
        fn_total += fn

    results.append([
        t,
        fp_total/runs,
        fn_total/runs
    ])

In [24]:
# Diplay Ouput results
df_q4 = pd.DataFrame(
    results,
    columns=[
        "Hash Functions (t)",
        "Avg False Positives",
        "Avg False Negatives"
    ]
)

df_q4

,Hash Functions (t),Avg False Positives,Avg False Negatives
0,50,302.4,2.8
1,100,94.0,1.2
2,200,33.2,1.2


Q4 Results Conclusion :

*   MinHash signatures were generated for each user based on the movies they rated
*   Similarity threshold of 0.5 was used for classification of user pairs as similar or dissimilar
*   For each value of t = 50, 100, 200, the experiment was repeated five times with different random hash functions.
*   False positives occur when MinHash predicts similarity ≥ 0.5 but the exact Jaccard similarity is below 0.5 while False negatives occur when MinHash predicts similarity < 0.5 but the true similarity is ≥ 0.5.
*   As the no of hash functions increase there is a reduction in both false positives as well as false negatives thus increasing the number of hash functions improves the accuracy of MinHash estimates.

Q5 LSH on MovieLens dataset

Break up the signature table into ( b ) bands with ( r ) hash functions per band and implement Locality Sensitive Hashing to find candidate pairs with a similarity of at least 0.6.

Experiment with:
Report the number of false positives and false negatives, taking the average over 5 runs.

How do these numbers change if we want a similarity of at least 0.8?
( r = 5, b = 10 ) for the table with the 50 hash functions
( r = 5, b = 20 ) for the table with the 100 hash functions
( r = 5, b = 40 ) and ( r = 10, b = 20 ) for the table with the 200 hash function

In [25]:
# Q5 Solution Code
# Import the libraries
import pandas as pd
import random
from itertools import combinations

t = 200
r = 5
b = 40
runs = 5
thresholds = [0.6, 0.8]

print("LSH parameters -> r =", r, ", b =", b, ", signature length =", t)

# Precompute true similar pairs (using exact Jaccard)
true_pairs = {}

for tau in thresholds:
    true_pairs[tau] = {
        pair for pair, sim in exact_sim.items()
        if sim >= tau
    }

print("True pairs ≥ 0.6:", len(true_pairs[0.6]))
print("True pairs ≥ 0.8:", len(true_pairs[0.8]))

# Accumulators
fp_totals = {tau: 0 for tau in thresholds}
fn_totals = {tau: 0 for tau in thresholds}
candidate_total = 0

# For LSH Experiments
for run in range(runs):

    random.seed(42 + run)
    hash_funcs = generate_hash_functions(t)

    # Compute MinHash signatures
    signatures = {
        u: minhash_signature(user_movies_list[u], hash_funcs)
        for u in users
    }

    # Build LSH buckets
    buckets = {}

    for u in users:
        sig = signatures[u]

        for band in range(b):
            start = band * r
            end = start + r

            band_sig = tuple(sig[start:end])
            key = (band, band_sig)

            buckets.setdefault(key, []).append(u)

    # Generate candidate pairs
    candidate_pairs = set()

    for bucket_users in buckets.values():
        if len(bucket_users) > 1:
            for pair in combinations(bucket_users, 2):
                candidate_pairs.add(tuple(sorted(pair)))
    print("Run", run + 1, "candidate pairs:", len(candidate_pairs))

    candidate_total += len(candidate_pairs)

    # Evaluate FP / FN for each threshold

    for tau in thresholds:

        # False Positives
        fp = sum(
            1 for pair in candidate_pairs
            if exact_sim[pair] < tau
        )

        # False Negatives
        fn = sum(
            1 for pair in true_pairs[tau]
            if pair not in candidate_pairs
        )

        fp_totals[tau] += fp
        fn_totals[tau] += fn

# Final Results Table

results = []

for tau in thresholds:

    results.append([
        tau,
        candidate_total / runs,
        fp_totals[tau] / runs,
        fn_totals[tau] / runs
    ])

pairs_06 = [(p,s) for p,s in exact_sim.items() if s >= 0.6]
pairs_08 = [(p,s) for p,s in exact_sim.items() if s >= 0.8]

print("\nPairs ≥ 0.6:", len(pairs_06))
print("Pairs ≥ 0.8:", len(pairs_08))

print("\nPairs ≥ 0.6:")
for p,s in pairs_06:
    print(p, "similarity = ",round(s,5))

print("\nPairs ≥ 0.8:")
for p,s in pairs_08:
    print(p, "similarity = ",round(s,5))

df_q5 = pd.DataFrame(
    results,
    columns=[
        "Similarity Threshold",
        "Avg Candidate Pairs",
        "Avg False Positives",
        "Avg False Negatives"
    ]
)

print("\nFinal Q5 Results")
df_q5

LSH parameters -> r = 5 , b = 40 , signature length = 200
True pairs ≥ 0.6: 3
True pairs ≥ 0.8: 1
Run 1 candidate pairs: 2630
Run 2 candidate pairs: 1947
Run 3 candidate pairs: 3987
Run 4 candidate pairs: 4976
Run 5 candidate pairs: 3311

Pairs ≥ 0.6: 3
Pairs ≥ 0.8: 1

Pairs ≥ 0.6:
(328, 788) similarity =  0.67296
(408, 898) similarity =  0.83871
(489, 587) similarity =  0.62992

Pairs ≥ 0.8:
(408, 898) similarity =  0.83871

Final Q5 Results


,Similarity Threshold,Avg Candidate Pairs,Avg False Positives,Avg False Negatives
0,0.6,3370.2,3367.2,0.0
1,0.8,3370.2,3369.2,0.0


In [26]:
# Run the experiment with differnt combinations of r and b values as given in question

import pandas as pd
import random
from itertools import combinations

runs = 5
thresholds = [0.6,0.8]

configs = [
    (50,5,10),
    (100,5,20),
    (200,5,40),
    (200,10,20)
]

results = []

# precompute true similar pairs
true_pairs = {
    tau:{pair for pair,sim in exact_sim.items() if sim>=tau}
    for tau in thresholds
}

for t,r,b in configs:

    print("\nRunning configuration: t =",t,", r =",r,", b =",b)

    fp_totals = {tau:0 for tau in thresholds}
    fn_totals = {tau:0 for tau in thresholds}

    for run in range(runs):

        random.seed(42+run)

        hash_funcs = generate_hash_functions(t)

        signatures = {
            u:minhash_signature(user_movies_list[u],hash_funcs)
            for u in users
        }

        buckets = {}

        for u in users:
            sig = signatures[u]

            for band in range(b):
                start = band*r
                end = start+r

                band_sig = tuple(sig[start:end])
                key = (band,band_sig)

                buckets.setdefault(key,[]).append(u)

        candidate_pairs = set()

        for bucket_users in buckets.values():
            if len(bucket_users)>1:
                for pair in combinations(bucket_users,2):
                    candidate_pairs.add(tuple(sorted(pair)))

        for tau in thresholds:

            fp = sum(
                1 for pair in candidate_pairs
                if exact_sim[pair] < tau
            )

            fn = sum(
                1 for pair in true_pairs[tau]
                if pair not in candidate_pairs
            )

            fp_totals[tau] += fp
            fn_totals[tau] += fn

    for tau in thresholds:

        results.append([
            t,r,b,
            tau,
            fp_totals[tau]/runs,
            fn_totals[tau]/runs
        ])

df_lsh_configs = pd.DataFrame(
    results,
    columns=[
        "t",
        "r",
        "b",
        "Similarity Threshold",
        "Avg False Positives",
        "Avg False Negatives"
    ]
)

# df_lsh_configs

table_06 = df_lsh_configs[df_lsh_configs["Similarity Threshold"] == 0.6] \
            .sort_values("t") \
            .reset_index(drop=True)

table_08 = df_lsh_configs[df_lsh_configs["Similarity Threshold"] == 0.8] \
            .sort_values("t") \
            .reset_index(drop=True)

print("\nResults for Similarity Threshold τ = 0.6\n")
display(table_06)

print("\nResults for Similarity Threshold τ = 0.8\n")
display(table_08)



Running configuration: t = 50 , r = 5 , b = 10

Running configuration: t = 100 , r = 5 , b = 20

Running configuration: t = 200 , r = 5 , b = 40

Running configuration: t = 200 , r = 10 , b = 20

Results for Similarity Threshold τ = 0.6



,t,r,b,Similarity Threshold,Avg False Positives,Avg False Negatives
0,50,5,10,0.6,786.6,0.2
1,100,5,20,0.6,2088.0,0.0
2,200,5,40,0.6,3367.2,0.0
3,200,10,20,0.6,14.8,1.2



Results for Similarity Threshold τ = 0.8



,t,r,b,Similarity Threshold,Avg False Positives,Avg False Negatives
0,50,5,10,0.8,788.4,0.0
1,100,5,20,0.8,2090.0,0.0
2,200,5,40,0.8,3369.2,0.0
3,200,10,20,0.8,15.6,0.0


Q5 Results Conclusion :

Our results on LSH experiments show that the choice of banding parameters significantly affects the number of false positives and false negatives. Configurations with smaller row size (r=5) and more bands, b produce a large number of candidate pairs, resulting in higher false positives but very few false negatives.

 Increasing the row size ( r = 10) reduces false positives considerably but may slightly increase false negatives.

 For the higher similarity threshold 𝜏 = 0.8, no false negatives were observed because only one highly similar pair exists in the dataset. Overall, the configuration t=200,r=10,b=20 provides a better balance between reducing false positives and maintaining accuracy oof detection